In [ ]:
pip install pandas beautifulsoup4 lxml selenium webdriver-manager


In [ ]:

pip install -U bottleneck


In [ ]:
pip install --upgrade pip
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter
python macrotrends_quarterly_all.py


In [ ]:
pip install selenium-wire

In [ ]:
pip install blinker


In [ ]:
pip install --upgrade pip


In [ ]:
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter


In [ ]:
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter


In [ ]:
# -*- coding: utf-8 -*-
"""
Macrotrends | Selenium + JQXGrid 스크롤 수거 방식
- Income / Balance Sheet / Cash Flow / Key Financial Ratios
- 수평 스크롤을 자동 수행하여 전체 분기 데이터를 수집
- 결과: TICKER_quarterly_full.xlsx (각 시트에 저장)
"""

import os, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome import ChromeDriverManager  # <- 이걸로 바꿈

# ==== 설정 ====
TICKER = "TSLA"
SLUGS = {"TSLA": "tesla"}        # 다른 티커는 여기에 추가: "AAPL":"apple"
CATEGORIES = {
    "income_statement":    "income-statement?freq=Q",
    "balance_sheet":       "balance-sheet?freq=Q",
    "cash_flow":           "cash-flow-statement?freq=Q",
    "key_financial_ratios":"key-financial-ratios?freq=Q",
}
BASE = "https://www.macrotrends.net"
START_DATE = pd.Timestamp("2009-06-30")
SAVE_PATH = f"{TICKER}_quarterly_full.xlsx"
HEADLESS = False

# ==== 헬퍼 함수들 ====
def to_num(s):
    try:
        s = str(s).replace(",", "").replace("$", "").strip()
        if s in ["", "-", "—"]:
            return pd.NA
        if s.endswith("%"):
            return float(s[:-1]) / 100
        return float(s)
    except:
        return pd.NA

def build_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    opts.page_load_strategy = "eager"
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def wait_ready(driver):
    WebDriverWait(driver, 30).until(lambda d: d.execute_script("return document.readyState") == "complete")
    time.sleep(2.0)

# 스크롤하면서 page_source 수집
def scroll_and_collect_html(driver, scroll_container_selector):
    """수평 스크롤을 좌→우로 자동 수행하며, 
       각 스텝별 page_source를 누적 리턴"""
    html_snapshots = []
    container = driver.find_element(By.CSS_SELECTOR, scroll_container_selector)
    scroll_width = driver.execute_script("return arguments[0].scrollWidth", container)
    client_width = driver.execute_script("return arguments[0].clientWidth", container)
    # 스크롤 단위 (반면 client_width 정도)
    step = int(client_width * 0.8) or 200
    for pos in range(0, scroll_width, step):
        driver.execute_script("arguments[0].scrollLeft = arguments[1]", container, pos)
        time.sleep(0.8)  # 렌더링 대기
        html_snapshots.append(driver.page_source)
    return html_snapshots

def parse_jqx_from_html(html):
    """html (페이지 source 또는 스냅샷)에서 데이터를 추출하는 함수"""
    import pandas as pd
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "lxml")
    # 헤더 (각 분기 날짜)
    header_spans = soup.select(".jqx-grid-column-header span")
    headers = [sp.get_text(strip=True) for sp in header_spans]
    headers = headers[1:]  # 첫 건은 항목명 컬럼
    # 행
    rows = soup.select("div[role='row']")
    records = []
    for r in rows[1:]:  # 첫 row는 header row
        cells = r.select("div[role='gridcell']")
        texts = [c.get_text(strip=True) for c in cells]
        if texts:
            label = texts[0]
            vals = [to_num(t) for t in texts[1:1+len(headers)]]
            records.append((label, vals))
    return headers, records

# ==== 메인 스크래핑 함수 ====
def scrape_category_full(driver, url):
    driver.get(url)
    wait_ready(driver)
    time.sleep(1.0)
    snapshots = scroll_and_collect_html(driver, ".jqx-grid-content")
    seen_labels = {}
    for html in snapshots:
        headers, recs = parse_jqx_from_html(html)
        for label, vals in recs:
            if label not in seen_labels:
                seen_labels[label] = vals
            else:
                # 덧붙이되 중복 위치 회피
                existing = seen_labels[label]
                for i, v in enumerate(vals):
                    if i >= len(existing):
                        existing.append(v)
                    elif existing[i] is pd.NA and v is not pd.NA:
                        existing[i] = v
    data = [(label, seen_labels[label]) for label in seen_labels]
    df = pd.DataFrame([ [label] + vals for label, vals in data ], columns=["LineItem"] + headers)
    return df

# ==== 실행부 ====
def main():
    slug = SLUGS.get(TICKER.upper())
    if not slug:
        raise RuntimeError("SLUGS dict에 회사 slug를 추가하세요")

    driver = build_driver()
    try:
        with pd.ExcelWriter(SAVE_PATH, engine="xlsxwriter") as writer:
            for name, tail in CATEGORIES.items():
                print(f"[{name}] scraping ...")
                url = f"{BASE}/stocks/charts/{TICKER}/{slug}/{tail}"
                df = scrape_category_full(driver, url)
                print(f"  → rows={len(df)}, cols={len(df.columns)}")
                df.to_excel(writer, sheet_name=name[:31], index=False)
                time.sleep(1.0)
        print("\n[SAVED] ", os.path.abspath(SAVE_PATH))
    finally:
        driver.quit()

if __name__ == "__main__":
    main()


In [ ]:
pip install -U selenium webdriver-manager


In [ ]:
# -*- coding: utf-8 -*-
"""
Macrotrends | 분기 재무 데이터 수집기 (Income, Balance Sheet, Cash Flow, Key Ratios)
- 입력: TICKER (예: TSLA, AAPL, NVDA)
- 기간: 2009-06-30 ~ 현재
- 출력: <script> 내 JSON -> 우선 사용, 실패 시 DIV 그리드 파싱
- 저장: <TICKER>_quarterly_all.xlsx (시트: income_statement, balance_sheet, cash_flow, key_financial_ratios)
주의: 사이트 약관/트래픽을 존중하세요.
"""

import os, re, time, io, sys, json
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# ===== 사용자 설정 =====
TICKER = "TSLA"   # <- 여기만 바꾸면 됩니다 (예: "AAPL", "NVDA")
START_DATE = pd.Timestamp("2009-06-30")
END_DATE   = pd.Timestamp("2100-12-31")  # 필요 시 조정

# ===== 공통 =====
HEADLESS = False  # 막힐 경우 False(창 표시)가 더 안정적
SAVE_PATH = f"{TICKER}_quarterly_all.xlsx"

def build_driver(headless=HEADLESS):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("--lang=en-US,en,ko-KR,ko")
    # 일반 브라우저처럼 보이기
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    return driver

def wait_ready(driver, extra_sleep=2.5):
    WebDriverWait(driver, 30).until(lambda d: d.execute_script("return document.readyState") == "complete")
    # 쿠키/동의 배너 닫기(있을 때만)
    for sel in ["#onetrust-accept-btn-handler",".fc-button.fc-cta-consent","button[aria-label='Accept all']"]:
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, sel)
            if btns:
                btns[0].click()
                time.sleep(0.5)
        except Exception:
            pass
    time.sleep(extra_sleep)

def resolve_company_slug(driver, ticker):
    """
    /stocks/charts/<TICKER> 로 접속하면 실제 회사 슬러그가 붙은 경로로 이동됨.
    그 최종 URL에서 슬러그를 추출 (예: 'https://.../stocks/charts/TSLA/tesla' -> 'tesla')
    """
    base = f"https://www.macrotrends.net/stocks/charts/{ticker}"
    driver.get(base)
    wait_ready(driver, 2.5)
    url = driver.current_url.strip("/")
    parts = url.split("/")
    # .../stocks/charts/TICKER/<slug>  구조
    slug = parts[-1]
    # 혹시 마지막 토큰이 티커라면 한 단계 위 토큰 사용
    if slug.upper() == ticker.upper() and len(parts) >= 2:
        slug = parts[-2]
    return slug

def to_num(s):
    if s is None or s == "":
        return pd.NA
    s = str(s).strip()
    if s in ["-", "—"]:
        return pd.NA
    s = s.replace(",", "").replace("$", "")
    if s.endswith("%"):
        try:
            return float(s[:-1]) / 100.0
        except:
            return pd.NA
    try:
        return float(s)
    except:
        return s

def tidy_matrix(columns, rows):
    """
    columns: ["2015-12-31", ...] (가정: 최신~과거 또는 반대)
    rows: list of {"name": "Revenue", "data": [.. same length as columns ..]}
    -> DataFrame 형태로 변환 + 날짜 필터 + 최신→과거 정렬
    """
    df = pd.DataFrame([[r.get("name","")] + r.get("data",[]) for r in rows],
                      columns=["LineItem"] + columns)
    # 날짜 정렬/필터
    # 일부 columns가 문자열일 수 있음
    col_dates = []
    for c in df.columns[1:]:
        try:
            d = pd.to_datetime(c, errors="coerce")
        except:
            d = pd.NaT
        col_dates.append((c, d))
    # 유효한 날짜만
    col_dates = [(c,d) for c,d in col_dates if pd.notna(d)]
    col_dates = [x for x in col_dates if START_DATE <= x[1] <= END_DATE]
    # 최신→과거
    col_dates.sort(key=lambda x: x[1], reverse=True)
    ordered_cols = ["LineItem"] + [c for c,_ in col_dates]
    df = df[ordered_cols]
    # 값 숫자화
    for c in df.columns[1:]:
        df[c] = df[c].map(to_num)
    return df

def parse_from_scripts(html):
    """
    페이지 <script> 내용에서 columns/rows JSON 블록을 찾아 파싱
    예상 패턴 예시:
       var data = {"columns":["2024-12-31",...], "rows":[{"name":"Revenue","data":[...]}]}
    """
    soup = BeautifulSoup(html, "lxml")
    scripts = soup.find_all("script")
    for sc in scripts:
        txt = sc.string or sc.text
        if not txt: 
            continue
        # 가장 일반적인 패턴들 탐색
        m = re.search(r'(\{[^{}]*"columns"\s*:\s*\[[^\]]+\][^{}]*"rows"\s*:\s*\[[\s\S]+?\]\s*\})', txt)
        if not m:
            m = re.search(r'(\{[^{}]*"rows"\s*:\s*\[[\s\S]+?\][^{}]*"columns"\s*:\s*\[[^\]]+\][^{}]*\})', txt)
        if m:
            try:
                blob = m.group(1)
                data = json.loads(blob)
                cols = [str(c) for c in data["columns"]]
                rows = data["rows"]
                return tidy_matrix(cols, rows)
            except Exception:
                continue
    return None

def parse_from_div_grid(html):
    """
    <table>이 없는 DIV 그리드에서 텍스트 기반으로 헤더(날짜)와 값 매트릭스를 재구성 (백업)
    """
    soup = BeautifulSoup(html, "lxml")

    # 날짜 헤더 후보 수집
    texts = [x.get_text(" ", strip=True) for x in soup.find_all(True)]
    date_tokens = [t for t in texts if re.fullmatch(r"\d{4}-\d{2}-\d{2}", t)]
    # 빈도수 높은 순으로 정리
    from collections import Counter
    cnt = Counter(date_tokens)
    headers = [t for t,_ in cnt.most_common()]
    # 유효 날짜 필터 및 정렬
    col_dates = [(h, pd.to_datetime(h, errors="coerce")) for h in headers]
    col_dates = [(h,d) for h,d in col_dates if pd.notna(d) and START_DATE <= d <= END_DATE]
    col_dates.sort(key=lambda x: x[1], reverse=True)
    headers = [h for h,_ in col_dates]
    if not headers:
        raise ValueError("날짜 헤더를 찾지 못했습니다(백업 경로 실패).")

    # 라인아이템: 왼쪽 목록의 링크 텍스트(항목명) 위주로 추출
    labels = []
    for a in soup.select("a"):
        s = a.get_text(strip=True)
        if s and not re.search(r"\d", s) and len(s) > 2:
            labels.append(s)
    # 고유화 & 순서 유지
    seen=set(); uniq=[]
    for s in labels:
        if s not in seen:
            seen.add(s); uniq.append(s)
    # 과도한 라벨 제거: 너무 일반적인 메뉴/탭 이름 제외
    drop_keywords = ["Income Statement","Balance Sheet","Cash Flow","Key Financial Ratios",
                     "Prices","Financials","Revenue & Profit","Assets & Liabilities",
                     "Margins","Other Ratios","Other Metrics","View Quarterly Reports",
                     "Format","Search for ticker or company name"]
    labels = [s for s in uniq if all(k.lower() not in s.lower() for k in drop_keywords)]

    # 값 후보(숫자/%, $)
    number_like = [t for t in texts if re.fullmatch(r"[\-\$]?\d[\d,\.]*%?", t)]
    # 간단한 휴리스틱: 각 라벨이 나온 인덱스 이후의 숫자들을 헤더 수만큼 배정
    # (정확도 향상을 위해 같은 텍스트 리스트 위치를 사용)
    def first_index(seq, token):
        for i,x in enumerate(seq):
            if x == token: return i
        return -1

    data_rows=[]
    for label in labels:
        idx = first_index(texts, label)
        if idx == -1:
            continue
        # 라벨 근처에서 숫자 헤더 수만큼 추출
        window = texts[idx: idx + 800]  # 넉넉한 창
        vals = [t for t in window if re.fullmatch(r"[\-\$]?\d[\d,\.]*%?", t)]
        vals = vals[:len(headers)]
        if len(vals) < int(len(headers)*0.4):
            continue
        data_rows.append([label] + vals)

    # DataFrame 구성
    df = pd.DataFrame(data_rows, columns=["LineItem"] + headers)
    for c in df.columns[1:]:
        df[c] = df[c].map(to_num)
    return df

def fetch_category(driver, ticker, slug, slug_tail):
    """
    slug_tail e.g. 'income-statement?freq=Q'
    """
    url = f"https://www.macrotrends.net/stocks/charts/{ticker}/{slug}/{slug_tail}"
    driver.get(url)
    wait_ready(driver, 3.0)

    html = driver.page_source
    # 1) 스크립트 내 JSON 시도
    df = parse_from_scripts(html)
    if df is not None:
        return df
    # 2) 백업: DIV 그리드 파싱
    return parse_from_div_grid(html)

def main():
    driver = build_driver()
    try:
        slug = resolve_company_slug(driver, TICKER)
        print(f"[INFO] TICKER={TICKER} | company slug='{slug}'")

        categories = {
            "income_statement":    "income-statement?freq=Q",
            "balance_sheet":       "balance-sheet?freq=Q",
            "cash_flow":           "cash-flow-statement?freq=Q",
            "key_financial_ratios":"key-financial-ratios?freq=Q",
        }

        with pd.ExcelWriter(SAVE_PATH, engine="xlsxwriter") as writer:
            for cat, tail in categories.items():
                print(f"[{cat}] fetching ...")
                df = fetch_category(driver, TICKER, slug, tail)
                # 열 이름에서 중복 날짜 제거(가능시)
                seen=set(); cols=[]
                for c in df.columns:
                    if c not in seen:
                        seen.add(c); cols.append(c)
                df = df[cols]
                df.to_excel(writer, sheet_name=cat[:31], index=False)
                print(f" -> sheet '{cat}' rows={len(df)} cols={len(df.columns)}")
                time.sleep(2.0)

        abs_path = os.path.abspath(SAVE_PATH)
        print(f"\n[SAVED] {abs_path}")

    finally:
        try: driver.quit()
        except: pass

if __name__ == "__main__":
    # 설치가 안되어 있으면 다음 실행:
    # pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter
    main()


In [ ]:
# 8/22 코드

In [ ]:
# -*- coding: utf-8 -*-
"""
Macrotrends | Selenium + JQXGrid 스크롤 수거 방식
- Income / Balance Sheet / Cash Flow / Key Financial Ratios
- 수평 스크롤을 자동 수행하여 전체 분기 데이터를 수집
- 결과: TICKER_quarterly_full.xlsx (각 시트에 저장)
"""

import os, time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ==== 설정 ====
TICKER = "TSLA"
SLUGS = {"TSLA": "tesla"}  # 다른 티커는 여기에 추가: "AAPL":"apple"
CATEGORIES = {
    "income_statement": "income-statement?freq=Q",
    "balance_sheet": "balance-sheet?freq=Q",
    "cash_flow": "cash-flow-statement?freq=Q",
    "key_financial_ratios": "key-financial-ratios?freq=Q",
}
BASE = "https://www.macrotrends.net"
START_DATE = pd.Timestamp("2009-06-30")
SAVE_PATH = f"{TICKER}_quarterly_full.xlsx"
HEADLESS = False

# ==== 헬퍼 함수들 ====
def to_num(s):
    try:
        s = str(s).replace(",", "").replace("$", "").strip()
        if s in ["", "-", "—"]:
            return pd.NA
        if s.endswith("%"):
            return float(s[:-1]) / 100
        return float(s)
    except:
        return pd.NA

def build_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    opts.page_load_strategy = "eager"

    driver_path = ChromeDriverManager().install()
    return webdriver.Chrome(service=Service(driver_path), options=opts)

def wait_ready(driver):
    WebDriverWait(driver, 30).until(lambda d: d.execute_script("return document.readyState") == "complete")
    time.sleep(2.0)

def scroll_and_collect_html(driver, scroll_container_selector):
    html_snapshots = []
    container = driver.find_element(By.CSS_SELECTOR, scroll_container_selector)
    scroll_width = driver.execute_script("return arguments[0].scrollWidth", container)
    client_width = driver.execute_script("return arguments[0].clientWidth", container)
    step = int(client_width * 0.8) or 200
    for pos in range(0, scroll_width, step):
        driver.execute_script("arguments[0].scrollLeft = arguments[1]", container, pos)
        time.sleep(0.8)
        html_snapshots.append(driver.page_source)
    return html_snapshots

def parse_jqx_from_html(html):
    soup = BeautifulSoup(html, "lxml")
    header_spans = soup.select(".jqx-grid-column-header span")
    headers = [sp.get_text(strip=True) for sp in header_spans][1:]
    rows = soup.select("div[role='row']")
    records = []
    for r in rows[1:]:
        cells = r.select("div[role='gridcell']")
        texts = [c.get_text(strip=True) for c in cells]
        if texts:
            label = texts[0]
            vals = [to_num(t) for t in texts[1:1+len(headers)]]
            records.append((label, vals))
    return headers, records

def scrape_category_full(driver, url):
    driver.get(url)
    wait_ready(driver)
    time.sleep(1.0)
    snapshots = scroll_and_collect_html(driver, ".jqx-grid-content")
    seen_labels = {}
    for html in snapshots:
        headers, recs = parse_jqx_from_html(html)
        for label, vals in recs:
            if label not in seen_labels:
                seen_labels[label] = vals
            else:
                existing = seen_labels[label]
                for i, v in enumerate(vals):
                    if i >= len(existing):
                        existing.append(v)
                    elif existing[i] is pd.NA and v is not pd.NA:
                        existing[i] = v
    data = [(label, seen_labels[label]) for label in seen_labels]
    df = pd.DataFrame([[label] + vals for label, vals in data], columns=["LineItem"] + headers)
    return df

# ==== 실행부 ====
def main():
    slug = SLUGS.get(TICKER.upper())
    if not slug:
        raise RuntimeError("SLUGS dict에 회사 slug를 추가하세요")

    driver = build_driver()
    try:
        with pd.ExcelWriter(SAVE_PATH, engine="xlsxwriter") as writer:
            for name, tail in CATEGORIES.items():
                print(f"[{name}] scraping ...")
                url = f"{BASE}/stocks/charts/{TICKER}/{slug}/{tail}"
                df = scrape_category_full(driver, url)
                print(f"  → rows={len(df)}, cols={len(df.columns)}")
                df.to_excel(writer, sheet_name=name[:31], index=False)
                time.sleep(1.0)
        print("\n[SAVED] ", os.path.abspath(SAVE_PATH))
    finally:
        driver.quit()

if __name__ == "__main__":
    main()


In [ ]:
pip uninstall webdriver-manager


In [ ]:
pip install webdriver-manager --upgrade
